# FSTIC Notebook (Student-Friendly)

Use the widgets below to select files and run analysis without typing paths.

## 1) Setup

In [17]:
from pathlib import Path
import glob
import os
import pandas as pd

import fstic

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as exc:
    raise RuntimeError('ipywidgets is required. Run ./setup_student.command and reopen this notebook.') from exc

PROJECT_ROOT = Path.cwd()
SPEECH_DIR = PROJECT_ROOT / 'speech'
DEFAULT_OUTPUT_DIR = PROJECT_ROOT / 'output' / 'notebook-runs'
DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMON_EXTENSIONS = ['wav', 'mp3', 'ogg', 'flac', 'aac', 'wma', 'm4a', 'aiff', 'opus']

def list_audio_files(folder: Path):
    files = []
    for ext in COMMON_EXTENSIONS:
        files.extend(folder.glob(f'*.{ext}'))
    return sorted(files)

audio_files = list_audio_files(SPEECH_DIR)
print(f'Project root: {PROJECT_ROOT}')
print(f'Speech dir: {SPEECH_DIR}')
print(f'Default output: {DEFAULT_OUTPUT_DIR}')
print(f'Audio files found: {len(audio_files)}')

Project root: /Users/tate.carson/Library/CloudStorage/OneDrive-DakotaStateUniversity/Courses/Spring 2026/forensics/FSTIC
Speech dir: /Users/tate.carson/Library/CloudStorage/OneDrive-DakotaStateUniversity/Courses/Spring 2026/forensics/FSTIC/speech
Default output: /Users/tate.carson/Library/CloudStorage/OneDrive-DakotaStateUniversity/Courses/Spring 2026/forensics/FSTIC/output/notebook-runs
Audio files found: 4


## 2) Single File (Widget)

In [18]:
single_file = widgets.Dropdown(
    options=[str(p) for p in audio_files],
    description='File:',
    layout=widgets.Layout(width='95%')
)
window_ms = widgets.IntText(value=500, description='Window ms:')
hop_ms = widgets.IntText(value=250, description='Hop ms:')
make_pdf = widgets.Checkbox(value=False, description='Generate PDF')
run_single = widgets.Button(description='Run Single File', button_style='success')
single_out = widgets.Output()

def on_run_single(_):
    with single_out:
        clear_output()
        path = single_file.value
        if not path:
            print('No file selected.')
            return
        ok, sti = fstic.process_audio_file(path, str(DEFAULT_OUTPUT_DIR), window_ms.value, hop_ms.value, create_pdf=make_pdf.value)
        if ok and sti is not None:
            print(f'Success: {Path(path).name} STI={sti:.3f}')
        else:
            print('Processing failed.')

run_single.on_click(on_run_single)
display(single_file, widgets.HBox([window_ms, hop_ms, make_pdf]), run_single, single_out)

Dropdown(description='File:', layout=Layout(width='95%'), options=('/Users/tate.carson/Library/CloudStorage/On…

Button(button_style='success', description='Run Single File', style=ButtonStyle())

Output()

## 3) Folder Batch (Widget)

In [ ]:
folder_text = widgets.Text(value=str(SPEECH_DIR), description='Folder:', layout=widgets.Layout(width='95%'))
batch_window = widgets.IntText(value=500, description='Window ms:')
batch_hop = widgets.IntText(value=250, description='Hop ms:')
batch_pdf = widgets.Checkbox(value=False, description='Generate PDF')
run_batch = widgets.Button(description='Run Batch', button_style='warning')
batch_out = widgets.Output()

def on_run_batch(_):
    with batch_out:
        clear_output()
        folder = Path(folder_text.value)
        if not folder.is_dir():
            print('Folder not found.')
            return
        files = list_audio_files(folder)
        if not files:
            print('No supported audio files found.')
            return
        rows = []
        for fp in files:
            ok, sti = fstic.process_audio_file(str(fp), str(DEFAULT_OUTPUT_DIR), batch_window.value, batch_hop.value, create_pdf=batch_pdf.value)
            rows.append({
                'filename': fp.name,
                'sti_mean': round(float(sti), 3) if ok and sti is not None else None,
                'success': bool(ok),
            })
        df = pd.DataFrame(rows)
        display(df)

run_batch.on_click(on_run_batch)
display(folder_text, widgets.HBox([batch_window, batch_hop, batch_pdf]), run_batch, batch_out)

## 4) Compare Two Files (Widget)

In [ ]:
file_options = [str(p) for p in audio_files]
compare_a = widgets.Dropdown(options=file_options, description='File A:', layout=widgets.Layout(width='95%'))
compare_b = widgets.Dropdown(options=file_options, description='File B:', layout=widgets.Layout(width='95%'))
cmp_window = widgets.IntText(value=500, description='Window ms:')
cmp_hop = widgets.IntText(value=250, description='Hop ms:')
cmp_pdf = widgets.Checkbox(value=False, description='Generate PDF')
run_compare = widgets.Button(description='Run Compare', button_style='info')
compare_out = widgets.Output()

def on_run_compare(_):
    with compare_out:
        clear_output()
        if not compare_a.value or not compare_b.value:
            print('Select both files.')
            return
        ok, a, b = fstic.compare_two_audio_files(compare_a.value, compare_b.value, str(DEFAULT_OUTPUT_DIR), cmp_window.value, cmp_hop.value, create_pdf=cmp_pdf.value)
        if ok:
            print(f'Success: A={a:.3f}, B={b:.3f}')
        else:
            print('Comparison failed.')

run_compare.on_click(on_run_compare)
display(compare_a, compare_b, widgets.HBox([cmp_window, cmp_hop, cmp_pdf]), run_compare, compare_out)

## 5) View Generated Files

In [ ]:
generated = sorted(DEFAULT_OUTPUT_DIR.glob('*'))
for p in generated:
    print(p)